In [ ]:
import torch
from transformers import T5EncoderModel, T5Tokenizer
from tqdm import tqdm
import numpy as np

# load ProtT5
model_name = "Rostlab/prot_t5_xl_uniref50"
tokenizer = T5Tokenizer.from_pretrained(model_name, do_lower_case=False)
model = T5EncoderModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()


def read_fasta(file):
    seqs = {}
    with open(file) as f:
        seq_id = None
        seq = []
        for line in f:
            line=line.strip()
            if line.startswith(">"):
                if seq_id:
                    seqs[seq_id] = "".join(seq)
                seq_id = line[1:].split()[0]
                seq=[]
            else:
                seq.append(line)
        if seq_id:
            seqs[seq_id] = "".join(seq)
    return seqs


fasta_file = "cath-dataset-nonredundant-S40-v4_3_0.fa"
sequences = read_fasta(fasta_file)

embeddings = {}

for seq_id, seq in tqdm(sequences.items()):

    seq = " ".join(list(seq))
    seq = seq.replace("U","X").replace("Z","X").replace("O","X")

    ids = tokenizer(seq, return_tensors="pt")
    input_ids = ids["input_ids"].to(device)
    attention_mask = ids["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    embedding = outputs.last_hidden_state[0]

    protein_embedding = embedding.mean(dim=0).cpu().numpy()

    embeddings[seq_id] = protein_embedding


np.save("cath_s40_prott5_embeddings.npy", embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/11.3G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/11.3G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
T5EncoderModel LOAD REPORT from: Rostlab/prot_t5_xl_uniref50
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
 84%|████████▍ | 26833/31885 [1:02:15<12:54,  6.52it/s]